# Task 1 - Member 4: Independence (5.9) and Validation (6)

**Problem.** Which events about the generated smart buildings are independent of each other, and do independent calculation routes (probability rule, direct count, Monte Carlo, bootstrap) give the same probabilities?

**Objectives.** (1) Test independence for 12 event pairs. (2) Validate the results of Members 2 and 3 three ways. (3) Measure uncertainty (bootstrap) and sensitivity (top 5% removed, other thresholds). (4) Save tables and charts for Member 5.

**Steps.** Set-up -> 5.9 independence table and lift chart -> 6.1 three-way validation -> 6.2 counting check -> 6.3 bootstrap -> 6.4 sensitivity -> hand-off to Member 5.

Random seed 42 everywhere. `df_raw` is never modified. Results describe the 200 accepted generated buildings, not real buildings.

## Set-up

This cell works in two situations:
- **Merged master notebook** (Members 1 and 3 already ran): it reuses `df`, `A`, `Bx`, `Pc`, `m3_results`.
- **Stand-alone** (fresh Colab): it loads the three CSVs and rebuilds the same `df` and event columns as Member 1.

It then adds two things that only Member 4 needs: event `N7` (7 AHUs configured) and `df_trim` (the 5% buildings with the most zones removed).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import comb, perm, isclose
from pathlib import Path
from scipy.stats import chi2_contingency
from IPython.display import display

EVENTS = ["A_large", "B_extra_ahu", "F2", "F3", "F4", "O2"]

def rebuild_member1_df():
    """Same steps as Member 1 (Sections 1, 3, 4). Used only if df is not already in memory."""
    need = ["buildings_200_features.csv", "rejected_seeds.csv", "feature_catalog.csv"]
    folders = [Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/stats_project")]
    data_dir = next((p for p in folders if (p / need[0]).exists()), None)
    if data_dir is None:
        from google.colab import files
        print("Upload buildings_200_features.csv now (rejected_seeds / feature_catalog are optional here)")
        files.upload(); data_dir = Path.cwd()
    raw = pd.read_csv(data_dir / need[0])                      # raw table is never modified
    d = raw.copy()
    d.columns = [c.split("__", 1)[1] for c in d.columns]        # drop only the first prefix
    d = d.drop(columns=[c for c in d.columns if d[c].nunique(dropna=False) == 1])   # 17 constants
    d["A_large"]     = d.n_zones > d.n_zones.median()           # strict >, median = 64
    d["B_extra_ahu"] = d.ahus_added_beyond_config > 0
    for k in (2, 3, 4):
        d[f"F{k}"] = d["floor_plan.num_floors"] == k
    d["O2"] = d["simulation.occupants_per_zone"] == 2
    return raw, d

if "df" not in globals() or not all(e in df.columns for e in EVENTS):
    df_raw, df = rebuild_member1_df()
    print("Stand-alone mode: rebuilt df from the CSV")
else:
    print("Using Member 1's df from this notebook")

# --- Member 4's own additions (df_raw untouched; N7 added to the analysis frame only) ---
med = df.n_zones.median()                          # 64
df["N7"] = df["ahu.num_ahus"] == 7                 # 7 AHUs were configured
cut = df.n_zones.quantile(0.95)                    # 157.7 zones
df_trim = df[df.n_zones <= cut].copy()             # 5% of buildings with the most zones removed
n = len(df)

print("df:", df.shape, "| df_trim:", df_trim.shape, "| cut =", round(cut, 1))
counts = df[EVENTS + ["N7"]].sum()
print(counts.to_dict())
assert n == 200 and len(df_trim) == 190
assert counts.to_dict() == {"A_large": 98, "B_extra_ahu": 146, "F2": 74, "F3": 63,
                            "F4": 63, "O2": 108, "N7": 39}, "event counts differ from Member 1 - check df"

## 5.9 Independence

**Question.** Does knowing one fact about a building change the chance of another fact?

**Events and assumptions.** `A_large` (n_zones > 64, 98 buildings), `B_extra_ahu` (AHUs added beyond configured, 146), `F2/F3/F4` (2/3/4 floors: 74/63/63), `O2` (2 occupants per zone, 108), `N7` (7 AHUs configured, 39). The 200 accepted buildings are treated as equally likely draws from the generator; tolerance is 2 standard errors.

**Rule and why it fits.** Two events are independent if P(A and B) = P(A) x P(B). We compare the actual joint probability with this product. With only 200 rows the gap is never exactly 0, so we call a pair *dependent* only when the gap is bigger than 2 SE of the gap, where SE = sqrt( P(A)(1-P(A)) P(B)(1-P(B)) / n ). A chi-square p-value is a cross-check. Lift = P(A and B) / (P(A)P(B)); 1 means independent, above 1 means they occur together more than chance, below 1 less than chance.

In [ ]:
def indep(a, b, name):
    pa, pb, pab = a.mean(), b.mean(), (a & b).mean()
    pp = pa * pb                                                   # expected joint if independent
    se = (pa*(1-pa)*pb*(1-pb) / n) ** 0.5                          # standard error of the gap
    p_chi2 = chi2_contingency(pd.crosstab(a, b), correction=False)[1]   # cross-check only
    return dict(pair=name, P_A=pa, P_B=pb, P_AandB=pab, P_A_x_P_B=pp, diff=pab-pp,
                lift=pab/pp, two_SE=2*se,
                verdict="dependent" if abs(pab-pp) > 2*se else "consistent with independence",
                p_chi2=p_chi2)

pairs = [("A_large","B_extra_ahu"), ("F4","B_extra_ahu"), ("O2","B_extra_ahu"),
         ("F4","O2"), ("F4","N7"), ("O2","A_large"),
         ("F2","B_extra_ahu"), ("F3","B_extra_ahu"), ("F4","A_large"),
         ("N7","B_extra_ahu"), ("N7","A_large"), ("O2","N7")]
indep_table = pd.DataFrame([indep(df[a], df[b], f"{a} vs {b}") for a, b in pairs])
display(indep_table.round(4))

dep = set(indep_table.loc[indep_table.verdict == "dependent", "pair"])
assert dep == {"A_large vs B_extra_ahu", "F4 vs B_extra_ahu", "F2 vs B_extra_ahu",
               "F4 vs A_large", "N7 vs B_extra_ahu"}, dep
print("Dependent pairs:", sorted(dep))

**Interpretation (5.9).** Five pairs are dependent: A_large/F4/F2/N7 each with the AHU outcome or size. They all trace back to the generator's capacity rule (required AHUs = ceil(zones / 10) versus 3 to 7 configured AHUs). The seven consistent pairs are settings the generator draws separately (floors, occupants, configured AHUs) or things outside the AHU rule (occupants). "Consistent with independence" does **not** prove independence; 200 rows cannot prove it. These are associations in generated data, not causes. **Limitation:** the 2 SE rule is a rule of thumb, so we also report the chi-square p-value.

### Lift chart
Lift = P(A and B) / (P(A) x P(B)); the black line at 1 means independent.

In [ ]:
t = indep_table.sort_values("lift")
colors = ["#C8662B" if v == "dependent" else "#2E6F95" for v in t.verdict]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(t.pair, t.lift, color=colors, height=0.6)
ax.axvline(1, color="black", lw=1)
for y, l in enumerate(t.lift):
    ax.text(l + 0.02, y, f"{l:.2f}", va="center", fontsize=8)
ax.set(title="Lift per event pair (orange = dependent, blue = consistent with independence)",
       xlabel="lift = P(A and B) / (P(A) x P(B))   (1 = independent)")
plt.tight_layout(); plt.savefig("lift_chart.png", dpi=150); plt.show()

## 6. Validation

### 6.1 Three-way validation table

**Question.** Do independent routes give the same probabilities?

**Events and assumptions.** Same 200 rows, seed 42. Monte Carlo draws 100,000 buildings from the 200 *with replacement*.

**Rule and why it fits.** Each result is computed (1) by its probability rule, (2) by directly counting rows, (3) by Monte Carlo. Rule and direct count are the same number written two ways, so they must match to rounding (< 1e-12). Monte Carlo is random, so it differs by sampling noise only (about sqrt(p(1-p)/100000) = 0.0016 at most). If Member 3's `m3_results` is in memory, its stored values are checked against ours as well.

In [ ]:
def Pc(a, b):                       # P(a | b) = count(a and b) / count(b)
    return (a & b).sum() / b.sum()

A, Bx = df.A_large, df.B_extra_ahu
pF  = {k: df[f"F{k}"].mean() for k in (2, 3, 4)}
pBF = {k: Pc(Bx, df[f"F{k}"]) for k in (2, 3, 4)}
pB_total = sum(pBF[k] * pF[k] for k in (2, 3, 4))               # total probability
post_F4  = pBF[4] * pF[4] / pB_total                              # Bayes

rng = np.random.default_rng(42)                                   # seed 42 -> same numbers every run
sim = df.loc[rng.choice(df.index, 100_000)]                       # 100,000 draws with replacement
cx  = pd.crosstab(A, Bx)                                          # another route to P(B|A)
sA, sB = sim.A_large, sim.B_extra_ahu

items = [
 ("P(B) total probability",     pB_total, Bx.mean(), sB.mean()),
 ("P(B|A) conditional",         (A & Bx).mean()/A.mean(), Pc(Bx, A), Pc(sB, sA)),
 ("P(A and B) multiplication",  A.mean()*Pc(Bx, A), (A & Bx).mean(), (sA & sB).mean()),
 ("P(A or B) addition",         A.mean()+Bx.mean()-(A & Bx).mean(), (A | Bx).mean(), (sA | sB).mean()),
 ("P(F4|B) Bayes",              post_F4, Pc(df.F4, Bx), Pc(sim.F4, sB)),
 ("P(not B) complement",        1-Bx.mean(), (~Bx).mean(), (~sB).mean()),
]
val = pd.DataFrame(items, columns=["result", "rule", "direct_count", "monte_carlo"])
val["|rule-direct|"] = (val.rule - val.direct_count).abs()
val["|direct-MC|"]   = (val.direct_count - val.monte_carlo).abs()
display(val.round(5))
assert (val["|rule-direct|"] < 1e-12).all(), "rule and direct count must agree exactly - look for a bug"
assert (val["|direct-MC|"] < 0.005).all(), "Monte Carlo too far from direct count"
print("crosstab route P(B|A):", round(cx.loc[True, True] / cx.loc[True].sum(), 4))
print("largest Monte Carlo noise seen:", round(val["|direct-MC|"].max(), 5))

# Optional: compare with Member 3's hand-off values (only if his cells ran in this notebook)
if "m3_results" in globals():
    chk = {"P(B|A)": Pc(Bx, A), "P(A and B)": (A & Bx).mean(), "P(B)": Bx.mean(), "P(F4|B)": Pc(df.F4, Bx)}
    for k, v in chk.items():
        assert isclose(m3_results[k], v, abs_tol=1e-12), f"Member 3 value {k} differs"
    print("Member 3 hand-off values match the direct counts.")
else:
    print("m3_results not in memory (stand-alone mode): Member 3 check skipped.")

**Interpretation (6.1).** Rule and direct count agree exactly (difference 0 up to rounding) because both use the same 200 rows. Monte Carlo differs by at most a few thousandths, which is ordinary sampling noise. If rule and direct count ever differ, look for a wrong denominator, a partition that does not add to 200, different row sets (`df` vs `df_trim`), or different thresholds (`>64` vs `>=64`). **Limitation:** Monte Carlo only re-draws our own 200 rows, so it checks our code, not the real generator.

### 6.2 Counting check (without replacement)

**Question.** If 3 buildings are picked at random without replacement, what is the chance all 3 need extra AHUs?

**Rule and why it fits.** Combinations: C(146,3) / C(200,3), because the order of the 3 buildings does not matter and no building can be picked twice. The shortcut 0.73^3 assumes replacement, so it should be slightly different.

In [ ]:
exact = comb(146, 3) / comb(200, 3)
rng = np.random.default_rng(42)
b = df.B_extra_ahu.to_numpy()
hits = sum(b[rng.choice(200, 3, replace=False)].all() for _ in range(100_000))
print(f"exact C(146,3)/C(200,3)          = {exact:.4f}")
print(f"simulation (no replacement)      = {hits/100_000:.4f}")
print(f"with-replacement shortcut 0.73^3 = {0.73**3:.4f}  (gap {0.73**3-exact:+.4f})")
print("ordered triples 200P3 =", perm(200, 3), "| unordered C(200,3) =", comb(200, 3))
assert abs(exact - hits/100_000) < 0.005

**Interpretation (6.2).** The exact value (0.387) and the simulation agree within noise. The with-replacement shortcut is a little too high because drawing without replacement slightly lowers the chance of another B building each time. **Limitation:** the simulation has noise of about 0.002.

### 6.3 Bootstrap (uncertainty)

**Question.** How uncertain is P(B|A) because we only have 200 buildings?

**Rule and why it fits.** Bootstrap: redraw 200 buildings with replacement 2,000 times, recompute P(B|A) each time, and take the middle 95% (2.5th to 97.5th percentile). It needs no formula, which suits a probability that sits near 1.

In [ ]:
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    s = df.loc[rng.choice(df.index, len(df))]
    if s.A_large.sum() > 0:
        boot.append(Pc(s.B_extra_ahu, s.A_large))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"P(B|A) = {Pc(Bx, A):.4f}   95% bootstrap interval [{lo:.4f}, {hi:.4f}]   sd = {np.std(boot):.4f}")
plt.figure(figsize=(6, 3.4)); plt.hist(boot, bins=25, color="#2E6F95")
plt.axvline(lo, color="#C8662B", ls="--"); plt.axvline(hi, color="#C8662B", ls="--")
plt.title("Bootstrap distribution of P(B|A)"); plt.xlabel("P(B|A)"); plt.ylabel("count")
plt.tight_layout(); plt.savefig("bootstrap_hist.png", dpi=150); plt.show()
assert lo <= Pc(Bx, A) <= hi

**Interpretation (6.3).** P(B|A) = 97/98 = 0.990, with a 95% interval of about 0.967 to 1.000. The interval touches 1 because many resamples contain no large building without extra AHUs. So the true rate is very likely above 0.96. **Limitation:** the bootstrap only reflects the 200 accepted buildings, not the generator's rejection step.

### 6.4 Sensitivity (do our own choices change the answer?)

**Question.** Does removing the 5% biggest buildings, or moving the "large" threshold, change the main results?

**Rule and why it fits.** We recompute the same probabilities on `df_trim` (190 rows, zones <= 157.7) and for thresholds 50, 64, 80 and 100. Stable results mean the conclusion does not depend on one buildings group or on the median convention.

In [ ]:
print(f"cut = {cut:.1f} zones; rows {len(df)} -> {len(df_trim)}  ({len(df)-len(df_trim)} buildings removed)")
A2, B2 = df_trim.n_zones > med, df_trim.B_extra_ahu               # same threshold (64)
sens = pd.DataFrame({
    "full (200)": [Bx.mean(), Pc(Bx, A), Pc(Bx, ~A), Pc(df.F4, Bx), df.O2.mean()],
    "trimmed":    [B2.mean(), Pc(B2, A2), Pc(B2, ~A2), Pc(df_trim.F4, B2), df_trim.O2.mean()],
}, index=["P(B)", "P(B|A)", "P(B|not A)", "P(F4|B)", "P(O2)"])
sens["change"] = sens["trimmed"] - sens["full (200)"]
display(sens.round(4))
rows = []
for thr in (50, 64, 80, 100):
    a = df.n_zones > thr
    rows.append((thr, a.mean(), Pc(Bx, a), Pc(Bx, ~a)))
thr_table = pd.DataFrame(rows, columns=["zone threshold", "P(large)", "P(B|large)", "P(B|not large)"])
display(thr_table.round(3))

**Interpretation (6.4).** After removing the 10 buildings above 157.7 zones, P(B|A) barely moves (0.990 to 0.989) and P(B) falls by about 1 point (0.730 to 0.716) and P(F4|B) by about 4 points (0.397 to 0.360). At every threshold large buildings are far more likely to need extra AHUs than not-large ones. So the main conclusion does not depend on the 5% largest buildings or on choosing 64. **Limitation:** trimming changes the population, so the probabilities are for "buildings up to 157.7 zones", not for all accepted buildings.

### Save results for Member 5

In [ ]:
member4_results = dict(independence=indep_table, validation=val, boot_ci=(lo, hi), sensitivity=sens, thresholds=thr_table)
indep_table.round(4).to_csv("m4_independence.csv", index=False)
val.round(5).to_csv("m4_validation.csv", index=False)
sens.round(4).to_csv("m4_sensitivity.csv")
print("saved: m4_independence.csv, m4_validation.csv, m4_sensitivity.csv, lift_chart.png, bootstrap_hist.png")
print("Member 5 can read the dictionary member4_results (keys:", list(member4_results), ")")